# revert, reflog, worktree

모든 저장소는 임시 폴더에 만들며 실제 GitHub 계정이나 원격 저장소를 변경하지 않습니다.

## 명령을 실행하기 전에 알아둘 Git 상태

Git은 파일을 세 자리에서 봅니다. **작업 폴더(working tree)** 는 지금 편집하는 파일이 있는 자리입니다. **색인(index)** 은 다음 커밋에 넣을 파일 상태를 골라 두는 자리이며, 스테이징 영역(staging area)이라고도 합니다. **커밋(commit)** 은 색인에 고른 상태를 저장소 이력에 기록한 결과입니다. 따라서 `git add app.py`는 파일을 영구 저장하는 명령이 아니라 현재 `app.py`를 색인에 올리는 명령입니다. 이어서 `git commit -m "메시지"`를 실행해야 그 상태가 커밋으로 기록됩니다.

**HEAD** 는 현재 체크아웃한 커밋을 가리킵니다. 보통 HEAD는 `main` 같은 **브랜치(branch)** 이름을 가리키고, 브랜치 이름은 다시 최신 커밋을 가리킵니다. 새 커밋을 만들면 현재 브랜치가 새 커밋으로 이동합니다. 이 관계를 알고 나면 `HEAD~1`은 “현재 커밋의 첫 번째 부모 한 단계 전”, `origin/main`은 “마지막으로 확인한 원격 `main`의 위치”로 읽을 수 있습니다.

명령은 왼쪽부터 `git` → 하위 명령 → 옵션 → 대상으로 읽습니다. 예를 들어 `git diff --cached -- app.py`에서 `diff`는 차이를 보고, `--cached`는 색인과 현재 커밋을 비교하며, `--`는 뒤의 `app.py`를 옵션이 아닌 파일 경로로 구분합니다. 코드의 `git("diff", "--cached", "--", "app.py", cwd=repo)`는 터미널에서 같은 명령을 실행하도록 인자를 나눈 Python 표현입니다.


## 준비

정상 설정 파일과 첫 커밋을 만듭니다. 이 절의 세 도구는 같은 종류의 기능이 아닙니다. **revert**는 이미 공유한 커밋의 효과를 취소하는 새 커밋을 만듭니다. **reflog**는 이 로컬 저장소에서 HEAD나 브랜치가 이동한 기록을 보여 줍니다. **worktree**는 같은 저장소의 다른 브랜치를 별도 작업 폴더에 연결합니다. 문제 상황을 먼저 구분한 뒤 도구를 선택해야 합니다.

In [1]:
# 셀이 만든 저장소만 사용하는 임시 작업 폴더를 준비합니다.
import subprocess
import sys
import tempfile
from pathlib import Path
workspace = tempfile.TemporaryDirectory(prefix="git-lesson-")
root = Path(workspace.name)
# cwd는 명령을 실행할 폴더이며 check=True는 명령 실패를 예외로 알립니다.
def git(*args: str, cwd: Path | None = None, check: bool = True) -> str:
    """지정한 임시 저장소에서 Git 명령을 실행합니다."""
    result = subprocess.run(["git", *args], cwd=cwd or root,
                            text=True, capture_output=True, check=check)
    return result.stdout.strip() or result.stderr.strip()
def configure(repo: Path) -> None:
    """실습 저장소에만 작성자 정보를 설정합니다."""
    git("config", "user.name", "Git Learner", cwd=repo)
    git("config", "user.email", "learner@example.invalid", cwd=repo)

In [2]:
# 복구 실습용 저장소를 준비합니다.
repo = root / "recovery"
repo.mkdir()
git("init", "-b", "main", cwd=repo)
configure(repo)
(repo / "config.txt").write_text("discount=10\n", encoding="utf-8")
git("add", "config.txt", cwd=repo)
git("commit", "-m", "feat: 할인율 10 설정", cwd=repo)
good = git("rev-parse", "HEAD", cwd=repo)
print(git("log", "--oneline", "-1", cwd=repo))

5f23af9 feat: 할인율 10 설정


## 기본 실행

잘못된 공개 변경은 revert로 반대 변경을 새 커밋에 기록합니다. `git revert --no-edit <커밋>`에서 `<커밋>`은 취소할 대상이고, `--no-edit`는 Git이 만든 기본 되돌림 메시지를 편집기 없이 사용합니다. `discount=90` 커밋은 이력에 남고 새 커밋이 그 패치를 반대로 적용해 파일을 `discount=10`으로 되돌립니다.

반면 reset은 현재 브랜치가 가리키는 위치를 옮깁니다. 파일 결과가 같아 보일 수 있어도 이력의 모양이 다릅니다. 공유한 잘못을 고칠 때는 기존 커밋을 보존하는 revert가 협업자가 이미 받은 이력과 맞추기 쉽습니다.

In [3]:
# revert는 할인율을 되돌리는 새 커밋을 만들어 기존 이력을 유지합니다.
(repo / "config.txt").write_text("discount=90\n", encoding="utf-8")
git("add", "config.txt", cwd=repo)
git("commit", "-m", "fix: 할인율 90 설정", cwd=repo)
bad = git("rev-parse", "HEAD", cwd=repo)
git("revert", "--no-edit", bad, cwd=repo)
print((repo / "config.txt").read_text(encoding="utf-8").strip())
print(git("log", "--oneline", "-3", cwd=repo))

discount=10
cd9aa3c Revert "fix: 할인율 90 설정"
a4318f3 fix: 할인율 90 설정
5f23af9 feat: 할인율 10 설정


## worktree로 두 작업 폴더 사용

별도 브랜치를 별도 폴더에 연결합니다. `git worktree add -b fix/urgent <경로>`에서 `add`는 작업 폴더를 추가하고, `-b fix/urgent`는 새 브랜치를 만들며, 마지막 인자는 폴더 위치입니다. 기본 폴더는 `main`의 `config.txt`를 유지하고 추가 폴더는 `fix/urgent`에서 `hotfix.txt`를 커밋합니다. 두 폴더는 커밋 객체를 공유하지만 체크아웃한 브랜치와 작업 파일은 구분됩니다.

In [4]:
# 별도 worktree에서 긴급 수정 브랜치를 사용하고 기본 폴더와 비교합니다.
worktree = root / "urgent-fix"
git("worktree", "add", "-b", "fix/urgent", worktree.as_posix(), cwd=repo)
(worktree / "hotfix.txt").write_text("urgent=true\n", encoding="utf-8")
git("add", "hotfix.txt", cwd=worktree)
git("commit", "-m", "fix: 긴급 설정 추가", cwd=worktree)
print("기본 폴더 브랜치:", git("branch", "--show-current", cwd=repo))
print("추가 폴더 브랜치:", git("branch", "--show-current", cwd=worktree))
print(git("worktree", "list", "--porcelain", cwd=repo))

기본 폴더 브랜치: main
추가 폴더 브랜치: fix/urgent
worktree /private/var/folders/14/3sq03f6s3_7bs0tygfqqvc_c0000gn/T/git-lesson-w_ydtm_j/recovery
HEAD cd9aa3c6aa4a3ee40fb99d6e1206cfc184e33e3b
branch refs/heads/main

worktree /private/var/folders/14/3sq03f6s3_7bs0tygfqqvc_c0000gn/T/git-lesson-w_ydtm_j/urgent-fix
HEAD 127383a2cd62cee8049dd2fde458dddb30620035
branch refs/heads/fix/urgent


## 실패 재현

아직 공유하지 않은 커밋을 hard reset으로 지우면 일반 log에서 보이지 않습니다.

In [5]:
# 임시 저장소의 미공유 커밋만 이동시켜 일반 log에서 사라짐을 관찰합니다.
git("switch", "-c", "experiment", cwd=repo)
(repo / "idea.txt").write_text("recover me\n", encoding="utf-8")
git("add", "idea.txt", cwd=repo)
git("commit", "-m", "experiment: 복구할 아이디어", cwd=repo)
lost = git("rev-parse", "HEAD", cwd=repo)
git("reset", "--hard", "HEAD~1", cwd=repo)
visible = git("log", "--format=%H", "-3", cwd=repo)
print("일반 log에 있음:", lost in visible)
print("파일 존재:", (repo / "idea.txt").exists())
assert lost not in visible

일반 log에 있음: False
파일 존재: False


## 수정

reflog에서 이동 전 커밋을 찾아 새 브랜치 이름을 붙입니다.

In [6]:
# reflog의 커밋 메시지로 복구 대상을 찾아 새 브랜치 이름을 붙입니다.
reflog = git("reflog", "--format=%H%x09%gs", "-5", cwd=repo)
print(reflog)
assert lost in reflog
candidate = next(
    line.split("\t", 1)[0] for line in reflog.splitlines()
    if "commit: experiment: 복구할 아이디어" in line
)
git("branch", "recovered-idea", candidate, cwd=repo)
recovered = git("show", "recovered-idea:idea.txt", cwd=repo)
print("복구 내용:", recovered)
print("복구 브랜치:", git("rev-parse", "--short", "recovered-idea", cwd=repo))
assert candidate == lost
assert recovered == "recover me"

cd9aa3c6aa4a3ee40fb99d6e1206cfc184e33e3b	reset: moving to HEAD~1
3eb27dd440bcd8fd2e7993606a6cf93d61d621b4	commit: experiment: 복구할 아이디어
cd9aa3c6aa4a3ee40fb99d6e1206cfc184e33e3b	checkout: moving from main to experiment
cd9aa3c6aa4a3ee40fb99d6e1206cfc184e33e3b	revert: Revert "fix: 할인율 90 설정"
a4318f36540e3c35f940138f2cf614fe96ffb544	commit: fix: 할인율 90 설정
복구 내용: recover me


복구 브랜치: 3eb27dd
